# Explorando o Data Lakehouse

Este notebook é o laboratório exploratório da aula. Tudo já está conectado — você não precisa configurar host, porta, usuário ou senha de nada.

**Antes de começar**: rode o notebook `01_construir_pipeline.ipynb` até o fim — ele lê o Postgres "fonte", grava a camada bronze no MinIO e constrói silver e gold. Sem isso, as tabelas abaixo ainda não existem.

Enquanto ele roda, dá pra acompanhar visualmente em duas telas:
- **MinIO Console** (http://localhost:9001) — vendo os arquivos Parquet aparecerem em `lakehouse/bronze`, `/silver`, `/gold`.
- **Trino UI** (http://localhost:8082) — vendo as queries `CREATE TABLE ... AS SELECT` rodando.

In [ ]:
import lakehouse_lab as lh
import matplotlib.pyplot as plt

## 1. O armazenamento bruto (MinIO)

Por baixo de tudo, o lakehouse é só um bucket S3 com pastas. Vamos listar o que existe fisicamente na camada bronze:

In [ ]:
lh.list_layer("bronze")

## 2. Consultando com SQL via Trino

O Trino enxerga esses arquivos como tabelas de verdade, organizadas em schemas (`bronze`, `silver`, `gold`) dentro do catálogo `lakehouse`.

In [ ]:
lh.query("SHOW SCHEMAS FROM lakehouse")

In [ ]:
# Camada bronze: dado cru, exatamente como veio da fonte
lh.query("SELECT * FROM lakehouse.bronze.customers LIMIT 10")

## 3. Camada Silver — dado limpo e unificado

Pedidos + itens + produtos + clientes já vêm juntados num único modelo, pronto para análise:

In [ ]:
lh.query("SELECT * FROM lakehouse.silver.sales LIMIT 10")

## 4. Camada Gold — agregados prontos para consumo

In [ ]:
gold_por_dia = lh.query("SELECT * FROM lakehouse.gold.sales_by_day ORDER BY order_date")
gold_por_dia

In [ ]:
gold_por_dia.plot(x="order_date", y="receita", kind="line", marker="o", figsize=(10, 4), title="Receita por dia")
plt.tight_layout()
plt.show()

In [ ]:
gold_por_categoria = lh.query("SELECT * FROM lakehouse.gold.sales_by_category ORDER BY receita DESC")
gold_por_categoria.plot(x="product_category", y="receita", kind="bar", figsize=(8, 4), title="Receita por categoria", legend=False)
plt.tight_layout()
plt.show()

## 5. Bônus: federação — juntando a fonte ao vivo com o data lake

O Trino também enxerga o Postgres "fonte" diretamente, sem passar pelo lakehouse. Dá pra comparar o dado ao vivo com o que já foi processado:

In [ ]:
lh.query(f"SELECT * FROM {lh.pg_schema()}.customers LIMIT 5", catalog="postgres")

## 6. Para explorar em aula

- Insira um novo pedido direto no Postgres (`lh.postgres()` + um `INSERT`) e rode de novo as células do notebook `01_construir_pipeline.ipynb` — veja o `gold.sales_by_day` mudar.
- Abra o MinIO Console e compare o tamanho/quantidade de arquivos entre bronze e gold — por que gold tem menos dado?
- Escreva uma nova query de agregação (ex: receita por cliente) direto em `lh.query(...)`.
- Derrube tudo com `docker compose down -v` e suba de novo — o ambiente inteiro volta ao estado zero.